# Ride-Share Market Analysis

| Block | Abschnitt der Arbeit | Ausgabe |
|---|---|---|
| 1 Introduction | 5.1.1 | Tabelle 1, Parameter, Effekt-Matrizen, PSE und Krümmungsmatrizen |
| 2 FTL | 5.2 und 5.3.1 | Tabelle 2, Gradientenschranke und die Werte im Text zu Tabelle 2 |
| 3 Equality | 5.4.1 | Tabelle 4 und die Werte im Text zu Tabelle 4 |

Die Tabellen liegen in Tables. Alle übrigen Werte erscheinen als Ausgabe unter den Zellen.

### Imports und Ausgabeordner
Lädt die Spielklasse `ddrideshare` aus `utils/utilsrm.py` und legt den Ordner `Tables` an. `write_table` schreibt die Kopf- und Datenzeilen einer Tabelle in diesen Ordner.

In [1]:
import os
import sys
sys.path.insert(1, '../utils/')
from utilsrm import *
import numpy as np, pandas as pd
from numpy import linalg as la

tab_dir = '../Tables'                                     # Ordner fuer die LaTeX-Tabellen der Arbeit
os.makedirs(tab_dir, exist_ok=True)

def write_table(fname, lines):
    # schreibt Kopf- und Datenzeilen einer Tabelle der Arbeit (ohne LaTeX-Formatierung) nach Tables/fname
    path = os.path.join(tab_dir, fname)
    with open(path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print('Tabelle geschrieben:', os.path.normpath(path))

## 1 Introduction (Abschnitt 5.1.1)
Spiel, Parameter, Effekt-Matrizen, PSE und Krümmungsmatrizen des Ride-Share-Spiels.

### Spiel aufsetzen
Erzeugt das Spiel `g6` mit allen 11 Orten (`loc_lst`), der Box bis `maxx` = 80 und der Regularisierung `e6_lam`, dazu die Basisnachfrage der niedrigsten Preisstufe. Definiert erwartete Nachfrage, Verlust, Best Response und die Lücken einer Mischung (`e6_eps`) und berechnet den PSE `e6_xstar` über das lineare Gleichungssystem der Best-Response-Abbildung. Alle weiteren Zellen bauen auf dieser Zelle auf. Keine Ausgabe.

In [2]:
g6 = ddrideshare(loc_lst=list(range(11)), price_lst=list(range(3)), seed=2, lam=[2.0, 2.0], base=True,
                 params={'A1': [], 'A2': [], 'Ac1': [], 'Ac2': []}, maxx=80)
g6.setup_distribution()
e6_d = g6.d; e6_lam = np.array([2.0, 2.0]); e6_lo, e6_hi = g6.l[0], g6.u[0]
e6_qL = g6.ql_[:, :, 0].T; e6_qU = g6.qu_[:, :, 0].T
e6_zb1, e6_zb2 = e6_qL.mean(1), e6_qU.mean(1)

def e6_Ez(x):                                   # E[z | joint model x], shape (2,d)
    return np.vstack((e6_zb1 + g6.A1 @ x[0] + g6.Ac1 @ x[1],
                      e6_zb2 + g6.A2 @ x[1] + g6.Ac2 @ x[0]))
def e6_loss(Z, X):                              # l_i = -1/2<z_i,th_i> + lam_i/2||th_i||^2
    return np.array([-0.5 * Z[i] @ X[i] + 0.5 * e6_lam[i] * (X[i] @ X[i]) for i in range(2)])
def e6_G(x):                                    # one-step retraining = best response to D(x)
    return np.clip(e6_Ez(x) / (2 * e6_lam[:, None]), e6_lo, e6_hi)
def e6_eps(X):                                  # eps(mu)=max_i gap_i for a mixture X=(M,2,d)
    X = np.asarray(X); M = len(X); PRv = np.zeros(2); mZ = np.zeros((2, e6_d))
    for x in X:
        Z = e6_Ez(x); mZ += Z; PRv += e6_loss(Z, x)
    PRv /= M; mZ /= M
    star = np.clip(mZ / (2 * e6_lam[:, None]), e6_lo, e6_hi)
    gap = PRv - e6_loss(mZ, star)
    return dict(gap=gap, eps=gap.max(), eps_sum=gap.sum(), block=int(np.argmax(gap)), M=M)

# spectral radius of the (interior) retraining map + exact PSE via linear solve
e6_J = np.block([[g6.A1 / (2 * e6_lam[0]),  g6.Ac1 / (2 * e6_lam[0])],
                 [g6.Ac2 / (2 * e6_lam[1]), g6.A2 / (2 * e6_lam[1])]])
e6_rho = np.abs(la.eigvals(e6_J)).max()
e6_xstar = np.clip(la.solve(np.eye(2 * e6_d) - e6_J,
                   np.concatenate([e6_zb1 / (2 * e6_lam[0]), e6_zb2 / (2 * e6_lam[1])])
                   ).reshape(2, e6_d), e6_lo, e6_hi)

### Werte für Abschnitt 5.1.1 und Tabelle 1
Parameterraum, Regularisierung, Preisstufe und Anzahl der aufgezeichneten Tage, Form, Vorzeichen und Größenverhältnis der Effekt-Matrizen, der Spektralradius der Best-Response-Abbildung (zitiert am Ende von Abschnitt 5.1.2) und der PSE mit der Prüfung, ob er im Inneren der Box liegt und die Projektion inaktiv ist. Gibt Tabelle 1 aus und schreibt sie nach `Tables/Table1_ExpRS.tex`.

In [3]:
# Parameterraum, Regularisierung, Preisstufe, Anzahl der Tage
print("Parameterraum: Theta_i = [%g, %g]^%d,   tau = %s,   Preisstufe %d $ (niedrigste Preisstufe: %s),   %d aufgezeichnete Tage je Ort"
      % (e6_lo, e6_hi, e6_d, e6_lam, g6.prices_[0], bool(g6.prices_[0] == min(g6.prices_)), e6_qL.shape[1]))

# Effekt-Matrizen: diagonal, eigene Effekte < 0, Konkurrenzeffekte > 0
print("\nEffekt-Matrizen:")
for name, M in [('P_1   (g6.A1) ', g6.A1), ('P_2   (g6.A2) ', g6.A2), ('P^c_1 (g6.Ac1)', g6.Ac1), ('P^c_2 (g6.Ac2)', g6.Ac2)]:
    dg = np.diag(M)
    print("  %s  diagonal: %-5s  Diagonale von %+7.2f bis %+7.2f"
          % (name, np.allclose(M, np.diag(dg)), dg.min(), dg.max()))
t41_own = np.concatenate([np.diag(g6.A1), np.diag(g6.A2)])
t41_com = np.concatenate([np.diag(g6.Ac1), np.diag(g6.Ac2)])
t41_rat = np.abs(t41_com) / np.abs(t41_own)                 # |A^c_i| / |A_i| je Ort und Spieler
print("  Konkurrenzeffekt / eigener Effekt je Ort und Spieler: %.2f bis %.2f, im Mittel %.2f"
      % (t41_rat.min(), t41_rat.max(), t41_rat.mean()))

# Best-Response-Abbildung ohne Projektion: G(theta) = c + J theta
t41_c    = np.concatenate([e6_zb1 / (2 * e6_lam[0]), e6_zb2 / (2 * e6_lam[1])])
t41_xsol = la.solve(np.eye(2 * e6_d) - e6_J, t41_c).reshape(2, e6_d)   # (I-J) theta* = c, ohne Clipping
print("\nSpektralradius rho(J) = %.1f" % e6_rho)
print("I - J invertierbar: %s  (kleinster Singulaerwert %.3f)"
      % (bool(la.matrix_rank(np.eye(2 * e6_d) - e6_J) == 2 * e6_d), la.svd(np.eye(2 * e6_d) - e6_J, compute_uv=False).min()))
t41_u = e6_Ez(t41_xsol) / (2 * e6_lam[:, None])                         # Best Response auf theta* ohne Projektion
print("PSE theta*: Koordinaten von %.2f bis %.2f;  im Inneren von [%g, %g]: %s;  Projektion bei theta* inaktiv: %s"
      % (t41_xsol.min(), t41_xsol.max(), e6_lo, e6_hi, bool(np.all((t41_xsol > e6_lo) & (t41_xsol < e6_hi))),
         bool(np.array_equal(np.clip(t41_u, e6_lo, e6_hi), t41_u))))
print("Abstand ||G(theta*) - theta*|| = %.2e" % la.norm(e6_G(t41_xsol) - t41_xsol))

# --- Tabelle 1: Basisnachfrage, Diagonalen der Effekt-Matrizen und PSE je Ort ---
t41_names = ['%s (%d)' % (str(n).replace('University', 'Uni.'), j + 1) for j, n in enumerate(g6.locations_)]
t41_tab = pd.DataFrame({'zbar_1': e6_zb1, 'zbar_2': e6_zb2,
                        '(P_1)_jj': np.diag(g6.A1), '(P_2)_jj': np.diag(g6.A2),
                        '(P^c_1)_jj': np.diag(g6.Ac1), '(P^c_2)_jj': np.diag(g6.Ac2),
                        'theta*_1j': t41_xsol[0], 'theta*_2j': t41_xsol[1]}, index=t41_names)
print("\nTabelle 1:")
print(t41_tab.to_string(formatters={c: ('{:.3f}' if c.startswith('theta') else '{:.2f}').format for c in t41_tab.columns}))
write_table('Table1_ExpRS.tex',
            [r"Location $(j)$ & $\bar z_1$ & $\bar z_2$ & $(P_1)_{jj}$ & $(P_2)_{jj}$ & $(P^c_1)_{jj}$ & $(P^c_2)_{jj}$ & $\theta^*_{1,j}$ & $\theta^*_{2,j}$ \\"]
            + [r"%s & %.2f & %.2f & %.2f & %.2f & %.2f & %.2f & %.3f & %.3f \\" % ((name,) + tuple(row))
               for name, row in zip(t41_tab.index, t41_tab.values)])

Parameterraum: Theta_i = [-80, 80]^11,   tau = [2. 2.],   Preisstufe 10 $ (niedrigste Preisstufe: True),   14 aufgezeichnete Tage je Ort

Effekt-Matrizen:
  P_1   (g6.A1)   diagonal: True   Diagonale von  -77.22 bis  -43.79
  P_2   (g6.A2)   diagonal: True   Diagonale von  -67.15 bis  -47.81
  P^c_1 (g6.Ac1)  diagonal: True   Diagonale von  +23.90 bis  +33.57
  P^c_2 (g6.Ac2)  diagonal: True   Diagonale von  +21.89 bis  +38.61
  Konkurrenzeffekt / eigener Effekt je Ort und Spieler: 0.33 bis 0.77, im Mittel 0.51

Spektralradius rho(J) = 26.5
I - J invertierbar: True  (kleinster Singulaerwert 7.415)
PSE theta*: Koordinaten von 8.91 bis 15.63;  im Inneren von [-80, 80]: True;  Projektion bei theta* inaktiv: True
Abstand ||G(theta*) - theta*|| = 6.20e-14

Tabelle 1:
                        zbar_1 zbar_2 (P_1)_jj (P_2)_jj (P^c_1)_jj (P^c_2)_jj theta*_1j theta*_2j
Haymarket Square (1)    447.64 291.93   -43.79   -67.15      33.57      21.89    15.628     8.913
Back Bay (2)            434.14 

### Krümmungsmatrizen und Assumption 5
Stellt die Krümmungsmatrizen der beiden Spieler (`e8_Hi`) und die gemeinsame Krümmungsmatrix (`e8_H`) auf und gibt ihre kleinsten und größten Eigenwerte aus. Definiert außerdem `e8_parts`, das eine Mischung in Zentrum und Streuung zerlegt und daraus alle Spalten von Tabelle 4 berechnet. Zum Schluss das Beispiel aus Abschnitt 5.1.1: ein Paar um den PSE entlang des Eigenvektors zum kleinsten Eigenwert der Krümmungsmatrix von Spieler 1, mit Prüfung der Eigenvektor-Eigenschaft.

In [4]:
# --- Kruemmungsmatrizen H_i in geschlossener Form, Reihenfolge theta = (theta_1, theta_2) ---
# H_1, H_2 sind 2x2-Blockmatrizen mit 11x11-Bloecken, also 22x22
e8_n = 2 * e6_d; e8_I = np.eye(e6_d); e8_O = np.zeros((e6_d, e6_d))
e8_Hi = [np.block([[e6_lam[0]/2*e8_I - 0.5*g6.A1, -0.25*g6.Ac1], [-0.25*g6.Ac1.T, e8_O]]),
         np.block([[e8_O, -0.25*g6.Ac2.T], [-0.25*g6.Ac2, e6_lam[1]/2*e8_I - 0.5*g6.A2]])]
e8_H = e8_Hi[0] + e8_Hi[1]
e8_ev, e8_V = la.eigh(e8_H); e8_lmin, e8_lmax = e8_ev[0], e8_ev[-1]

def e8_parts(X):
    # Zerlegung von Lemma 1 fuer die uniforme Mischung ueber die Profile X = (M, 2, d)
    X = np.asarray(X, float); Y = X.reshape(len(X), -1)
    c = Y.mean(0); S = (Y - c).T @ (Y - c) / len(Y)            # Zentrum theta^c und Kovarianzmatrix Sigma
    r = e6_eps(X); rc = e6_eps(c.reshape(1, 2, e6_d))          # eps(phi) und eps(theta^c) direkt aus Definition 9
    HS = np.array([np.sum(Hi * S) for Hi in e8_Hi])            # <H_i, Sigma>
    return dict(M=len(X), tr=np.trace(S), eps=r['eps_sum'], eps_c=rc['eps_sum'], HS=HS.sum(),
                delta=r['eps_sum'] - rc['eps_sum'] - HS.sum(), bound=e8_lmin * np.trace(S),
                diff=HS.sum() - e8_lmin * np.trace(S), gap=r['gap'], delta_i=r['gap'] - rc['gap'] - HS)

# Assumption 5: H positiv definit, die H_i allein dagegen indefinit
for i in range(2):
    w = la.eigvalsh(e8_Hi[i])
    print("H_%d allein: lambda_min = %.2f, lambda_max = %.2f, %d von %d Eigenwerten negativ  ->  indefinit: %s"
          % (i + 1, w[0], w[-1], (w < 0).sum(), e8_n, bool(w[0] < 0 < w[-1])))
print("H = H_1 + H_2 (%dx%d): lambda_min = %.2f, lambda_max = %.2f  ->  positiv definit: %s"
      % (e8_n, e8_n, e8_lmin, e8_lmax, bool(e8_lmin > 0)))

# Paar theta* +- u, u = Eigenvektor zum kleinsten Eigenwert von H_1 (Laenge 1)
u = la.eigh(e8_Hi[0])[1][:, 0].reshape(2, e6_d)
r = e8_parts(np.stack([e6_xstar + u, e6_xstar - u]))
print("\nPaar theta* +- v:  ||v|| = %.2f,  v^T H_1 v = %.2f  ->  v Einheitseigenvektor zu lambda_min(H_1): %s"
      % (la.norm(u), u.ravel() @ e8_Hi[0] @ u.ravel(),
         bool(np.isclose(la.norm(u), 1) and np.allclose(e8_Hi[0] @ u.ravel(), la.eigvalsh(e8_Hi[0])[0] * u.ravel()))))
print("g_1 = %+.2f,  g_2 = %+.2f,  G_S = %.2f" % (r['gap'][0], r['gap'][1], r['eps']))

H_1 allein: lambda_min = -2.75, lambda_max = 40.83, 11 von 22 Eigenwerten negativ  ->  indefinit: True
H_2 allein: lambda_min = -2.90, lambda_max = 36.06, 11 von 22 Eigenwerten negativ  ->  indefinit: True
H = H_1 + H_2 (22x22): lambda_min = 13.69, lambda_max = 54.10  ->  positiv definit: True

Paar theta* +- v:  ||v|| = 1.00,  v^T H_1 v = -2.75  ->  v Einheitseigenvektor zu lambda_min(H_1): True
g_1 = -2.75,  g_2 = +27.99,  G_S = 25.24


## 2 FTL (Abschnitte 5.2 und 5.3.1)
100 unabhängige Follow-the-Leader-Läufe (Seeds 200 bis 299, Start im Nullprofil, je 8000 Runden), ausgewertet nach 100, 500, 2000 und 8000 Runden. Die gepoolte Mischung aller Läufe schätzt die gemittelte Trajektorien-Mischung, das Mittel des realisierten Regrets über die Läufe schätzt den erwarteten Regret.

### FTL und Kennzahlen je Lauf
`e7_run` spielt FTL für beide Spieler und gibt die gespielten Profile und die beobachtete Nachfrage zurück. `e7_stats` fasst einen Lauf für jede ausgewertete Rundenzahl zusammen (mittlerer gespielter Verlust, mittlere erwartete Nachfrage und realisierter Regret je Runde). `e7_gap` berechnet daraus die Lücken der gepoolten Mischung. Keine Ausgabe.

In [5]:
def e7_run(T, seed, BATCH=20):
    # FTL ab theta^1 = 0; gibt Iterierte X und realisierte Nachfrage Z zurueck
    np.random.seed(seed)
    X = np.zeros((T, 2, e6_d)); Z = np.zeros((T, 2, e6_d)); zs = np.zeros((2, e6_d))
    for t in range(1, T + 1):
        xt = g6.proj(np.zeros((2, e6_d))) if t == 1 else np.clip(zs/(2*e6_lam[:, None]*(t-1)), e6_lo, e6_hi)
        X[t-1] = xt
        Z[t-1] = np.vstack((g6.query_env_player(xt, 0, e6_qL, batch=BATCH),
                            g6.query_env_player(xt, 1, e6_qU, batch=BATCH)))
        zs += Z[t-1]
    return X, Z

def e7_stats(X, Z, Ts):
    # je Praefix T: Kennzahlen, aus denen sich g_i der gepoolten Mischung exakt berechnen laesst,
    # und der realisierte Regret_i(T)/T
    lam = e6_lam[None, :]
    EZ = np.stack([e6_zb1 + X[:, 0] @ g6.A1.T + X[:, 1] @ g6.Ac1.T,
                   e6_zb2 + X[:, 1] @ g6.A2.T + X[:, 0] @ g6.Ac2.T], 1)        # E[z_i | theta^t], (T,2,d)
    sq = np.einsum('tij,tij->ti', X, X)
    dep  = -0.5*np.einsum('tij,tij->ti', EZ, X) + 0.5*lam*sq                    # R_i(theta_i^t; theta^t)
    real = -0.5*np.einsum('tij,tij->ti', Z, X)  + 0.5*lam*sq                    # f_i^t(theta_i^t)
    out = []
    for T in Ts:
        zbar = Z[:T].mean(0)
        a = np.clip(zbar/(2*e6_lam[:, None]), e6_lo, e6_hi)                     # bester fester Parameter im Nachhinein
        best = -0.5*np.sum(zbar*a, 1) + 0.5*e6_lam*np.sum(a*a, 1)
        out.append(dict(dep=dep[:T].mean(0), mEZ=EZ[:T].mean(0), reg=real[:T].mean(0) - best))
    return out

def e7_gap(dep, mEZ):
    # g_i der gepoolten Mischung aus den gemittelten Kennzahlen (Definition 15):
    # erster Term = mittlerer gespielter Verlust, zweiter Term = Minimum ueber feste a_i in Theta_i
    star = np.clip(mEZ/(2*e6_lam[:, None]), e6_lo, e6_hi)
    return dep - e6_loss(mEZ, star)

### Gradientenschranke (Abschnitt 5.2)
Obere Schranke für die Gradienten der Rundenverluste je Spieler, gebildet aus dem größtmöglichen Betrag der Nachfrage je Ort innerhalb der Box.

In [6]:
# gleichmaessige Gradientenschranke G_i aus der groesstmoeglichen Nachfrage je Ort
e7_zmax = []
for i, (q, Pown, Pcom) in enumerate([(e6_qL, g6.A1, g6.Ac1), (e6_qU, g6.A2, g6.Ac2)]):
    sh = e6_hi*(np.abs(np.diag(Pown)) + np.abs(np.diag(Pcom)))
    e7_zmax.append(np.maximum(np.abs(q.min(1) - sh), np.abs(q.max(1) + sh)))
e7_Gi = np.array([np.sqrt(np.sum((0.5*e7_zmax[i] + e6_lam[i]*e6_hi)**2)) for i in range(2)])
print("Gradientenschranke: G_1 = %.1f,  G_2 = %.1f" % tuple(e7_Gi))

Gradientenschranke: G_1 = 13907.2,  G_2 = 13550.1


### 100 FTL-Läufe
Führt die Läufe aus und speichert je Lauf und Rundenzahl die Kennzahlen aus `e7_stats`. Die Kontrolle am Ende vergleicht die Lücken aus den Kennzahlen mit der direkten Berechnung über `e6_eps`; die Abweichung muss im Bereich von Rundungsfehlern liegen. Außerdem wird geprüft, ob jeder Lauf im Nullprofil beginnt. Dauert etwa 5 Minuten.

In [7]:
# --- 100 unabhaengige Laeufe, je 8000 Runden FTL ab theta^1 = 0 ---
e7_n, e7_Tmax, e7_seed0 = 100, 8000, 200
e7_Ts = [100, 500, 2000, 8000]; e7_K = len(e7_Ts)
e7_dep = np.zeros((e7_n, e7_K, 2)); e7_mEZ = np.zeros((e7_n, e7_K, 2, e6_d))
e7_reg = np.zeros((e7_n, e7_K, 2))
e7_runs = []                                                     # realisierte Mischungen fuer die Kontrolle unten
e7_x0 = True                                                     # beginnt jeder Lauf im Nullprofil?
print("%d Laeufe (Seeds %d-%d) mit je %d Runden, %d Ziehungen je Runde und Ort, Auswertung bei T = %s"
      % (e7_n, e7_seed0, e7_seed0 + e7_n - 1, e7_Tmax, e7_run.__defaults__[0], e7_Ts))
for s in range(e7_n):
    X, Z = e7_run(e7_Tmax, seed=e7_seed0 + s)
    e7_x0 &= bool(np.all(X[0] == 0))
    for k, r in enumerate(e7_stats(X, Z, e7_Ts)):
        e7_dep[s, k], e7_mEZ[s, k], e7_reg[s, k] = r['dep'], r['mEZ'], r['reg']
    if s < 5: e7_runs.append(X[:100])
    if (s + 1) % 25 == 0:
        print("  %d/%d Laeufe fertig" % (s + 1, e7_n))

# Kontrolle: Kennzahlen-Formel == e6_eps auf der zusammengelegten Mischung (5 Laeufe, T = 100)
e7_chk = e6_eps(np.concatenate(e7_runs))['gap'] - e7_gap(e7_dep[:5, 0].mean(0), e7_mEZ[:5, 0].mean(0))
print("Kontrolle gepoolte Luecke gegen e6_eps: max. Abweichung %.1e" % np.abs(e7_chk).max())
print("Start im Nullprofil in allen Laeufen: %s" % e7_x0)

100 Laeufe (Seeds 200-299) mit je 8000 Runden, 20 Ziehungen je Runde und Ort, Auswertung bei T = [100, 500, 2000, 8000]


  25/100 Laeufe fertig


  50/100 Laeufe fertig


  75/100 Laeufe fertig


  100/100 Laeufe fertig
Kontrolle gepoolte Luecke gegen e6_eps: max. Abweichung 5.8e-11
Start im Nullprofil in allen Laeufen: True


### Schätzer und gepaartes Bootstrap
Lücke der gepoolten Mischung, Toleranz aus dem mittleren Regret, ihre Differenz Delta sowie ein gepaartes Bootstrap über die Läufe mit 10 000 Ziehungen (Seed 7) für Standardfehler und 95%-Konfidenzintervalle. Keine Ausgabe.

In [8]:
def e7_est(idx):
    g = np.array([e7_gap(e7_dep[idx, k].mean(0), e7_mEZ[idx, k].mean(0)) for k in range(e7_K)])   # (K,2)
    R = e7_reg[idx].mean(0)                                                                        # (K,2)
    return g, R
e7_g, e7_R = e7_est(np.arange(e7_n))
e7_G, e7_eT = e7_g.max(1), np.maximum(0, e7_R.max(1))        # G(phibar_T) und eps_T = max{0, max_i E[Regret_i/T]}
e7_B = 10000; e7_rng = np.random.default_rng(7)
e7_bg = np.zeros((e7_B, e7_K, 2)); e7_bR = np.zeros((e7_B, e7_K, 2))
for b in range(e7_B):                                        # gepaartes Bootstrap ueber die Laeufe
    e7_bg[b], e7_bR[b] = e7_est(e7_rng.integers(0, e7_n, e7_n))
e7_bG, e7_beT = e7_bg.max(2), np.maximum(0, e7_bR.max(2))
e7_D = e7_G - e7_eT; e7_bD = e7_bG - e7_beT
e7_lo95, e7_hi95 = np.percentile(e7_bD, [2.5, 97.5], axis=0)

### Tabelle 2
Gibt die Tabelle lesbar aus (in Klammern die Bootstrap-Standardfehler) und schreibt Kopf- und Datenzeilen nach `Tables/Table2_FTL_RS.tex`.

In [9]:
print("Ride-Share: %d Laeufe (Seeds %d-%d), in Klammern Bootstrap-Standardfehler (%d Ziehungen)\n"
      % (e7_n, e7_seed0, e7_seed0 + e7_n - 1, e7_B))
print("  %6s | %24s | %24s | %10s | %24s" % ("T", "G(phibar_P)", "eps_T", "Delta", "95%-KI fuer Delta"))
for k, Tc in enumerate(e7_Ts):
    print("  %6d | %11.2f (%9.2f) | %11.2f (%9.2f) | %+10.2f | [%+9.2f, %+9.2f]"
          % (Tc, e7_G[k], e7_bG[:, k].std(ddof=1), e7_eT[k], e7_beT[:, k].std(ddof=1), e7_D[k], e7_lo95[k], e7_hi95[k]))

e7_lines = [r"$T$ & $G(\cmixap)$ & $\hat{\ce}_T$ & $\Delta$ & 95\% CI for $\Delta$ \\"]
for k, Tc in enumerate(e7_Ts):
    e7_lines.append(r"%d & $%.2f \ (%.2f)$ & $%.2f \ (%.2f)$ & $%+.2f$ & $[%+.2f, %+.2f]$ \\"
                    % (Tc, e7_G[k], e7_bG[:, k].std(ddof=1), e7_eT[k], e7_beT[:, k].std(ddof=1), e7_D[k], e7_lo95[k], e7_hi95[k]))
print()
write_table('Table2_FTL_RS.tex', e7_lines)

Ride-Share: 100 Laeufe (Seeds 200-299), in Klammern Bootstrap-Standardfehler (10000 Ziehungen)

       T |              G(phibar_P) |                    eps_T |      Delta |        95%-KI fuer Delta
     100 |   275432.14 (   238.78) |   275436.35 (   240.81) |      -4.21 | [   -24.12,    +16.26]
     500 |    55087.59 (    47.76) |    55088.42 (    48.16) |      -0.83 | [    -4.82,     +3.25]
    2000 |    13772.14 (    11.94) |    13772.35 (    12.04) |      -0.21 | [    -1.20,     +0.81]
    8000 |     3443.09 (     2.99) |     3443.15 (     3.01) |      -0.05 | [    -0.30,     +0.20]

Tabelle geschrieben: ../Tables/Table2_FTL_RS.tex


### Werte für Abschnitt 5.3.1
Auflösung des Tests (obere Konfidenzgrenze relativ zur Toleranz), relativer Abstand der beiden Seiten, die Aussage je Spieler und das Produkt aus Rundenzahl und Lücke.

In [10]:
# groesste mit den Daten vertraegliche Verletzung = obere KI-Grenze von Delta
print("Obere KI-Grenze von Delta relativ zu eps_T:", ", ".join("%.1e" % v for v in e7_hi95 / e7_eT),
      "  (bei T = %d absolut %.2f)" % (e7_Ts[-1], e7_hi95[-1]))
print("Abstand |G - eps_T| relativ zu eps_T:", ", ".join("%.1e" % v for v in np.abs(e7_D) / e7_eT))

print("\nje Spieler: g_i(phibar_P) - mittlerer Regret_i(T)/T mit 95%-Bootstrap-KI")
e7_in = True
for k, Tc in enumerate(e7_Ts):
    di = e7_bg[:, k] - e7_bR[:, k]; lo, hi = np.percentile(di, [2.5, 97.5], axis=0)
    e7_in &= bool(np.all((lo <= 0) & (hi >= 0)))
    print("  T = %5d:" % Tc, "  ".join("i=%d: g=%.2f R=%.2f  %+.3f [%+.3f, %+.3f]"
          % (i + 1, e7_g[k, i], e7_R[k, i], e7_g[k, i] - e7_R[k, i], lo[i], hi[i]) for i in range(2)))
print("Alle KIs je Spieler enthalten 0:", e7_in)

print("\nT * G(phibar_P):", ", ".join("%.4g (T = %d)" % (Tc * e7_G[k], Tc) for k, Tc in enumerate(e7_Ts)))

Obere KI-Grenze von Delta relativ zu eps_T: 5.9e-05, 5.9e-05, 5.9e-05, 5.9e-05   (bei T = 8000 absolut 0.20)
Abstand |G - eps_T| relativ zu eps_T: 1.5e-05, 1.5e-05, 1.5e-05, 1.5e-05

je Spieler: g_i(phibar_P) - mittlerer Regret_i(T)/T mit 95%-Bootstrap-KI
  T =   100: i=1: g=275432.14 R=275436.35  -4.213 [-24.120, +16.263]  i=2: g=228377.43 R=228377.38  +0.047 [-17.580, +17.654]
  T =   500: i=1: g=55087.59 R=55088.42  -0.832 [-4.821, +3.251]  i=2: g=45677.05 R=45677.03  +0.024 [-3.501, +3.542]
  T =  2000: i=1: g=13772.14 R=13772.35  -0.207 [-1.204, +0.812]  i=2: g=11419.58 R=11419.58  -0.000 [-0.879, +0.880]
  T =  8000: i=1: g=3443.09 R=3443.15  -0.052 [-0.301, +0.203]  i=2: g=2854.98 R=2854.98  +0.000 [-0.220, +0.221]
Alle KIs je Spieler enthalten 0: True

T * G(phibar_P): 2.754e+07 (T = 100), 2.754e+07 (T = 500), 2.754e+07 (T = 2000), 2.754e+07 (T = 8000)


## 3 Equality (Abschnitt 5.4.1)
Zerlegung der aggregierten Lücke für zehn Typen von Mischungen in die Lücke des Zentrums und den Streuungsterm, zusammen mit der unteren Schranke für den Streuungsterm. Je Typ 100 Mischungen; nur die PSE-Punktmasse und das Paar entlang des kleinsten Eigenvektors sind deterministisch. Die FTL-Mischungen stammen aus denselben 100 Läufen wie in Block 2 (Seeds 200 bis 299) und werden hier neu gerechnet.

### Erzeugung der Mischungen
Paare um den PSE (Eigenvektor zum kleinsten Eigenwert sowie 100 Zufallsrichtungen, Seed [8,1]), Gauß-Wolken aus je 200 Profilen um den PSE und um ein festes Zufallsprofil (Standardabweichung 1 mit Seed [8,2], 0.01 mit Seed [8,4], festes Profil aus Seed 1), die realisierten FTL-Mischungen nach 100 und 8000 Runden und Zufallsmischungen aus 2 bis 50 gleichverteilten Profilen (Seed [8,6]). Eine Wolke mit einem Profil außerhalb der Box würde verworfen und neu gezogen. Dauert etwa 5 Minuten.

In [11]:
def e8_ftl(T, seed, BATCH=20):
    # FTL ab theta^1 = 0, dieselben Iterierten wie e7_run in Block 2
    np.random.seed(seed); X = np.zeros((T, 2, e6_d)); zs = np.zeros((2, e6_d))
    for t in range(1, T + 1):
        xt = g6.proj(np.zeros((2, e6_d))) if t == 1 else np.clip(zs/(2*e6_lam[:, None]*(t-1)), e6_lo, e6_hi)
        X[t-1] = xt
        zs[0] += g6.query_env_player(xt, 0, e6_qL, batch=BATCH)
        zs[1] += g6.query_env_player(xt, 1, e6_qU, batch=BATCH)
    return X

e8_N = 100                                       # Mischungen je Typ, wie die 100 Laeufe in Block 2
e8_cseed = 1                                     # Seed fuer das feste Zufallsprofil theta~
e8_vmin = e8_V[:, 0].reshape(2, e6_d)            # Eigenvektor zu lambda_min(H), Laenge 1
# theta~: einmal gleichverteilt aus Theta gezogen (Abstand 10 zum Rand, damit alle Wolken-Profile in Theta liegen)
e8_xc = np.random.default_rng(e8_cseed).uniform(e6_lo + 10, e6_hi - 10, (2, e6_d))

def e8_unit(rng):
    v = rng.normal(size=(2, e6_d)); return v / la.norm(v)

def e8_clouds(center, s, k):
    # 100 Wolken aus je 200 Profilen ~ N(center, s^2 I), nur Wolken ganz in Theta; gleiches k = dieselben Zufallszahlen, nur anderes Zentrum
    global e8_rej
    rng = np.random.default_rng([8, k]); out = []
    while len(out) < e8_N:
        Z = rng.normal(0, s, (200, 2, e6_d))
        X = center + Z - Z.mean(0)                             # exakt zentrieren: theta^c = center, Sigma bleibt gleich
        if X.min() < e6_lo or X.max() > e6_hi:                 # Profil ausserhalb von Theta: Wolke verwerfen, neu ziehen
            e8_rej += 1; continue
        out.append(e8_parts(X))
    return out
e8_rej = 0                                      # Anzahl verworfener Wolken

e8_mix = {}                                      # Typ der Mischung -> Zerlegungen (e8_parts) aller Mischungen
e8_mix['theta* (PSE)']    = [e8_parts(e6_xstar[None])]                                     # deterministisch
e8_mix['theta* +- v_min'] = [e8_parts(np.stack([e6_xstar + e8_vmin, e6_xstar - e8_vmin]))]  # deterministisch
rng = np.random.default_rng([8, 1])
e8_mix['theta* +- v'] = [e8_parts(np.stack([e6_xstar + v, e6_xstar - v]))
                         for v in (e8_unit(rng) for _ in range(e8_N))]
e8_mix['Wolke um theta*, s = 1']    = e8_clouds(e6_xstar, 1.0, 2)
e8_mix['Wolke um theta*, s = 0.01'] = e8_clouds(e6_xstar, 0.01, 4)
e8_mix['Wolke um theta~, s = 1']    = e8_clouds(e8_xc, 1.0, 2)      # dieselben Wolken, um theta~ verschoben
e8_mix['Wolke um theta~, s = 0.01'] = e8_clouds(e8_xc, 0.01, 4)
e8_mix['FTL, T = 100'] = []; e8_mix['FTL, T = 8000'] = []
e8_bnd = []; e8_cum = []                         # je Lauf: letzte Iteration mit einem Preis am Rand, Anteile der ersten Iterationen an der Spur
for s in range(e8_N):                            # dieselben 100 Laeufe wie in Block 2 (Seeds 200-299)
    X = e8_ftl(8000, seed=200 + s)
    e8_mix['FTL, T = 100'].append(e8_parts(X[:100])); e8_mix['FTL, T = 8000'].append(e8_parts(X))
    e8_onb = np.any(np.abs(X) >= e6_hi, axis=(1, 2))                   # Iterationen mit mindestens einem Preis am Rand der Box
    e8_bnd.append(int(np.nonzero(e8_onb)[0].max()) + 1 if e8_onb.any() else 0)
    e8_Y = X.reshape(len(X), -1); e8_dev = np.sum((e8_Y - e8_Y.mean(0))**2, 1)   # quadrierter Abstand jeder Iteration zum Zentrum
    e8_cum.append(np.cumsum(e8_dev[:20]) / e8_dev.sum())                # Anteil der ersten 1, ..., 20 Iterationen an der Spur
    if (s + 1) % 25 == 0:
        print("  FTL: %d/%d Laeufe fertig" % (s + 1, e8_N))
rng = np.random.default_rng([8, 6])
e8_mix['Zufallsmischung'] = [e8_parts(rng.uniform(e6_lo, e6_hi, (int(rng.integers(2, 51)), 2, e6_d)))
                             for _ in range(e8_N)]
print("Verworfene Wolken: %d" % e8_rej)

  FTL: 25/100 Laeufe fertig


  FTL: 50/100 Laeufe fertig


  FTL: 75/100 Laeufe fertig


  FTL: 100/100 Laeufe fertig
Verworfene Wolken: 0


### Tabelle 4
Mittelwert und Standardabweichung je Typ, lesbar ausgegeben (mit der Anzahl der Mischungen je Typ). Kopf- und Datenzeilen werden nach `Tables/Table4_Set_RS.tex` geschrieben, Mittelwerte mit 7 und Standardabweichungen mit 6 Ziffern. Bei Typen mit nur einer Mischung entfällt die Standardabweichung.

In [12]:
e8_cols = ['G_S(phi)', 'G_S(delta_c)', '<H,Sigma>', 'lmin*Trace']
e8_keys = ['eps', 'eps_c', 'HS', 'bound']
e8_mean = pd.DataFrame({c: [np.mean([r[k] for r in rs]) for rs in e8_mix.values()]
                        for c, k in zip(e8_cols, e8_keys)}, index=list(e8_mix))
e8_sd   = pd.DataFrame({c: [np.std([r[k] for r in rs], ddof=1) if len(rs) > 1 else 0.0 for rs in e8_mix.values()]
                        for c, k in zip(e8_cols, e8_keys)}, index=list(e8_mix))

def e8_fmt(v, digits):
    # feste Ziffernzahl, z.B. 7 Ziffern: 0.003104, 171341.9, 1773067
    if abs(v) < 0.5 * 10.0**(1 - digits): v = 0.0
    dec = max(digits - len(str(int(abs(v)))), 0)
    s = "%.*f" % (dec, v)
    if sum(ch.isdigit() for ch in s) > digits and dec > 0:     # Aufrunden hat eine Stelle ergaenzt
        s = "%.*f" % (dec - 1, v)
    return s

e8_m = {name: [e8_fmt(e8_mean.loc[name, c], 7) for c in e8_cols] for name in e8_mix}    # Mittelwert: 7 Ziffern
e8_s = {name: [e8_fmt(e8_sd.loc[name, c], 6) for c in e8_cols] for name in e8_mix}      # SD: 6 Ziffern
print("Mittelwert (SD) je Typ der Mischung\n")
print("  %-26s | %6s | " % ("Typ", "Anzahl") + " | ".join("%19s" % c for c in e8_cols))
for name in e8_mix:
    print("  %-26s | %6d | " % (name, len(e8_mix[name]))
          + " | ".join("%19s" % ("%s (%s)" % (m, s)) for m, s in zip(e8_m[name], e8_s[name])))

e8_tex = {'theta* (PSE)': r'PSE $\dirac_{\cxps}$', 'theta* +- v_min': r'$\cxps\pm v_{\min}$', 'theta* +- v': r'$\cxps\pm v$',
          'Wolke um theta*, s = 1': r'Cloud $\mathcal N(\cxps,I)$',
          'Wolke um theta*, s = 0.01': r'Cloud $\mathcal N(\cxps,0.01^2I)$',
          'Wolke um theta~, s = 1': r'Cloud $\mathcal N(\cxf,I)$',
          'Wolke um theta~, s = 0.01': r'Cloud $\mathcal N(\cxf,0.01^2I)$',
          'FTL, T = 100': r'FTL $\cmixt(\cO)$, $T=100$', 'FTL, T = 8000': r'FTL $\cmixt(\cO)$, $T=8000$',
          'Zufallsmischung': 'random mixture'}
e8_lines = [r"Type of mixture & $\Gs(\cmix)$ & $\Gs(\dirac_{\cm})$ & $\langle\cCurv,\cCov\rangle$ & $\lambda_{\min}(\cCurv)\,\mathrm{Trace}(\cCov)$ \\"]
for name in e8_mix:
    e8_row = r"%s & %s \\" % (e8_tex[name], " & ".join(e8_m[name]))
    if len(e8_mix[name]) > 1:                                    # SD nur bei Typen mit mehreren Mischungen
        e8_row += r" & %s \\" % " & ".join("(%s)" % s for s in e8_s[name])
    e8_lines.append(e8_row)
print()
write_table('Table4_Set_RS.tex', e8_lines)

Mittelwert (SD) je Typ der Mischung

  Typ                        | Anzahl |            G_S(phi) |        G_S(delta_c) |           <H,Sigma> |          lmin*Trace
  theta* (PSE)               |      1 |  0.000000 (0.00000) |  0.000000 (0.00000) |  0.000000 (0.00000) |  0.000000 (0.00000)
  theta* +- v_min            |      1 |  13.68767 (0.00000) |  0.000000 (0.00000) |  13.68767 (0.00000) |  13.68767 (0.00000)
  theta* +- v                |    100 |  31.48608 (4.48203) |  0.000000 (0.00000) |  31.48608 (4.48203) |  13.68767 (0.00000)
  Wolke um theta*, s = 1     |    100 |  684.7485 (16.4585) |  0.000000 (0.00000) |  684.7485 (16.4585) |  299.6299 (6.41812)
  Wolke um theta*, s = 0.01  |    100 |  0.068252 (0.00167) |  0.000000 (0.00000) |  0.068252 (0.00167) |  0.029862 (0.00061)
  Wolke um theta~, s = 1     |    100 |   3628628 (16.4585) |   3627943 (0.00000) |  684.7485 (16.4585) |  299.6299 (6.41812)
  Wolke um theta~, s = 0.01  |    100 |   3627943 (0.00167) |   3627943 (0.00000)

### Werte für Abschnitt 5.4.1
Residuum der Zerlegung, Prüfung der unteren Schranke samt Verhältnis zur Schranke, Prüfung auf eine weitere PSME, Anteile von Zentrum und Streuung bei den Wolken und den FTL-Mischungen, Produkt aus Rundenzahl und Spur sowie die ersten FTL-Iterationen mit Preisen am Rand der Box.

In [13]:
e8_all = [r for name, rs in e8_mix.items() if name != 'theta* (PSE)' for r in rs]          # alle Mischungen ausser der PSE
e8_oth = [r for name, rs in e8_mix.items() if name not in ('theta* (PSE)', 'theta* +- v_min') for r in rs]
print("Theorem 2, %d Mischungen ohne PSE: groesstes relatives Residuum %.1e fuer G_S und %.1e fuer die einzelnen g_i"
      % (len(e8_all), max(abs(r['delta']) / r['eps'] for r in e8_all),
         max(np.max(np.abs(r['delta_i']) / np.abs(r['gap'])) for r in e8_all)))

# Lemma 2: Schranke in allen Mischungen, Gleichheit bei der PSE und entlang des kleinsten Eigenvektors?
print("\nLemma 2 gilt fuer alle Mischungen: %s" % all(r['diff'] >= -1e-9 * max(1.0, abs(r['HS'])) for r in e8_all))
for name in ['theta* (PSE)', 'theta* +- v_min']:
    r = e8_mix[name][0]
    print("%s: Abstand zur Schranke %.1e  ->  Schranke mit Gleichheit erreicht: %s"
          % (name, r['diff'], bool(abs(r['diff']) <= 1e-9 * max(1.0, abs(r['HS'])))))
e8_q = [r['HS'] / r['bound'] for r in e8_oth]
print("Verhaeltnis <H,Sigma> / Schranke fuer alle anderen Mischungen: %.2f bis %.2f" % (min(e8_q), max(e8_q)))

# Theorem 3: gibt es unter den getesteten Mischungen eine weitere PSME?
print("\nTheorem 3: G_S(phi) > 0 in jeder Mischung ausser der PSE: %s;  Mischungen mit g_i <= 0 fuer beide Spieler: %d von %d"
      % (all(r['eps'] > 0 for r in e8_all), sum(bool(np.all(r['gap'] <= 0)) for r in e8_all), len(e8_all)))
print("Paare theta* +- v mit negativer Luecke fuer einen Spieler: %d von %d"
      % (sum(bool(np.any(r['gap'] < 0)) for r in e8_mix['theta* +- v']), len(e8_mix['theta* +- v'])))

# Wolken: Zentrum nicht am PSE gegenueber Streuung um den PSE
print("\nWolken um theta~: Luecke des Zentrums %.2e, Anteil an G_S im Mittel %.3f%% (s = 1) und %.3f%% (s = 0.01)"
      % (e8_mean.loc['Wolke um theta~, s = 1', 'G_S(delta_c)'],
         100 * np.mean([r['eps_c'] / r['eps'] for r in e8_mix['Wolke um theta~, s = 1']]),
         100 * np.mean([r['eps_c'] / r['eps'] for r in e8_mix['Wolke um theta~, s = 0.01']])))
print("Streuungsterm der Wolken um theta~ gegenueber denen um theta*: max. relative Abweichung %.1e"
      % max(abs(a['HS'] / b['HS'] - 1) for s in ['1', '0.01']
            for a, b in zip(e8_mix['Wolke um theta~, s = ' + s], e8_mix['Wolke um theta*, s = ' + s])))
print("Wolken um theta*: von s = 1 auf s = 0.01 faellt G_S um den Faktor %.0f und die Spur der Kovarianzmatrix um den Faktor %.0f"
      % (e8_mean.loc['Wolke um theta*, s = 1', 'G_S(phi)'] / e8_mean.loc['Wolke um theta*, s = 0.01', 'G_S(phi)'],
         np.mean([r['tr'] for r in e8_mix['Wolke um theta*, s = 1']]) / np.mean([r['tr'] for r in e8_mix['Wolke um theta*, s = 0.01']])))

# FTL-Mischungen: Anteil der Streuung, Spur der Kovarianz und erste Iterationen
e8_sh = [100 * r['HS'] / r['eps'] for r in e8_mix['FTL, T = 8000']]
print("\nFTL, T = %d: Anteil der Streuung an G_S mindestens %.3f%%, Luecke des Zentrums im Mittel %.2f"
      % (e8_mix['FTL, T = 8000'][0]['M'], min(e8_sh), e8_mean.loc['FTL, T = 8000', 'G_S(delta_c)']))
print("FTL: T * Trace(Sigma_T) im Mittel", ", ".join("%.3g (T = %d)" % (np.mean([r['M'] * r['tr'] for r in e8_mix[n]]), e8_mix[n][0]['M'])
                                                     for n in ['FTL, T = 100', 'FTL, T = 8000']))
e8_k = max(e8_bnd)
print("Letzte Iteration mit einem Preis am Rand der Box: %d bis %d (Mittel %.1f);  Anteil der ersten %d Iterationen an der Spur bei T = %d: %.0f%%"
      % (min(e8_bnd), max(e8_bnd), np.mean(e8_bnd), e8_k, e8_mix['FTL, T = 8000'][0]['M'], 100 * np.mean([c[e8_k - 1] for c in e8_cum])))

Theorem 2, 801 Mischungen ohne PSE: groesstes relatives Residuum 4.1e-11 fuer G_S und 6.2e-11 fuer die einzelnen g_i

Lemma 2 gilt fuer alle Mischungen: True
theta* (PSE): Abstand zur Schranke 0.0e+00  ->  Schranke mit Gleichheit erreicht: True
theta* +- v_min: Abstand zur Schranke 1.8e-15  ->  Schranke mit Gleichheit erreicht: True
Verhaeltnis <H,Sigma> / Schranke fuer alle anderen Mischungen: 1.70 bis 3.20

Theorem 3: G_S(phi) > 0 in jeder Mischung ausser der PSE: True;  Mischungen mit g_i <= 0 fuer beide Spieler: 0 von 801
Paare theta* +- v mit negativer Luecke fuer einen Spieler: 0 von 100

Wolken um theta~: Luecke des Zentrums 3.63e+06, Anteil an G_S im Mittel 99.981% (s = 1) und 100.000% (s = 0.01)
Streuungsterm der Wolken um theta~ gegenueber denen um theta*: max. relative Abweichung 1.8e-14
Wolken um theta*: von s = 1 auf s = 0.01 faellt G_S um den Faktor 10033 und die Spur der Kovarianzmatrix um den Faktor 10034

FTL, T = 8000: Anteil der Streuung an G_S mindestens 99.996%, Lu